# Parametric Study - energy sweep

Run the same He -> Fe simulation at several beam energies and compare 
- the total vacancies produced per ion.
- the ion implantation depth.  

The pattern generalizes to any parameter sweep (material, fluence, displacement threshold, ...).

In [17]:
import numpy as np
import matplotlib.pyplot as plt

import opentrim

In [ ]:
def make_config(energy_eV, n_ions=10000):
    cfg = opentrim.Config()
    cfg.IonBeam.ion = opentrim.Element('He')
    cfg.IonBeam.energy_distribution.center = energy_eV

    cfg.Transport.flight_path_type = opentrim.FlightPath.Variable

    mat = opentrim.Material(); mat.id = 'Fe'; mat.density = 7.874
    atom = opentrim.Atom(); atom.element = opentrim.Element('Fe')
    atom.X = 1.0; atom.Ed = 40.0; atom.El = 3.0; atom.Es = 4.28; atom.Er = 40.0; atom.Rc = 0.946
    mat.composition.append(atom)
    cfg.Target.materials.append(mat)

    region = opentrim.Region(); region.id = 'slab'; region.material_id = 'Fe'
    region.origin = [0, 0, 0]; region.size = [12000, 12000, 12000]
    cfg.Target.regions.append(region)

    cfg.Target.cell_count = [120, 1, 1]
    cfg.Target.size = [12000, 12000, 12000]
    cfg.Simulation.electronic_stopping = opentrim.Stopping.SRIM13
    cfg.Simulation.simulation_type = opentrim.SimulationType.FullCascade
    # output base name allows only [0-9a-zA-Z_], so build it from integer keV
    cfg.Output.outfilename = f'He_{int(round(energy_eV / 1e3))}keV_Fe'
    cfg.Run.max_no_ions = n_ions
    cfg.Run.threads = 4
    cfg.Run.seed = 1
    cfg.validate()
    return cfg

In [ ]:
energies = [1e6, 2e6, 3e6, 5e6]   # eV
total_vac = []
impl_depth = []

for E in energies:
    sim = opentrim.Driver()
    sim.init(make_config(E))
    sim.run()
    sim.wait()
    info = opentrim.Info(sim)
    v, _ = info['tally']['damage_events']['Vacancies']
    impl, _ = info['tally']['damage_events']['Implantations']
    x_edges = info['target']['grid']['X']
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    total_vac.append(float(v.sum()))   # vacancies per ion, summed over depth
    impl_depth.append((x_centers @ impl[0,:,0,0])/impl[0,:,0,0].sum()/1000) # implantation depth (μm)
    print(f'{E/1e6:.0f} MeV -> {total_vac[-1]:.2f} vacancies/ion, Implantation depth: {impl_depth[-1]:.2f} μm')

In [ ]:
plt.figure(figsize=(7, 4))
ax = plt.gca()
ax.plot(np.array(energies) / 1e6, total_vac, 'o-', color='C0')
ax.set_xlabel('Beam energy (MeV)')
ax.set_ylabel('Total vacancies per ion', color='C0')
ax.tick_params(axis='y', colors='C0')
ax.set_title('He -> Fe: Damage & Implantation depth vs Energy')
ax.grid(True, alpha=0.3)
# right-side y-axis for implantation depth (μm)
ax2 = ax.twinx()
ax2.plot(np.array(energies) / 1e6, impl_depth, 's--', color='C1')
ax2.set_ylabel('Implantation depth (μm)', color='C1')
ax2.tick_params(axis='y', colors='C1')
plt.tight_layout()
plt.show()